In [ ]:
import os
import json
import shutil
os.chdir("../..")

In [ ]:
# STEP 1: MOVING TRAINING OUTPUTS MESSY WEIGHTS TO ORGANIZED LOGS
# extract outputs\training\ --> model_weights\training_logs\
# e.g. outputs\training\soft_constraint\run_YYYYMMDD_HHMMSS\hyperparams.json
# --> get "raw_csv": "development\datasets\binary_only\aci_set2.csv"
# --> get "seed": 21
# --> create model_weights\training_logs\seed_xx\<constraint>\constraint_xx.pt

input_root = "outputs/training"
output_root = "model_weights/training_logs"

for constraint in os.listdir(input_root):
    constraint_path = os.path.join(input_root, constraint)
    if not os.path.isdir(constraint_path):
        continue

    for run in os.listdir(constraint_path):
        run_path = os.path.join(constraint_path, run)
        if not os.path.isdir(run_path):
            continue

        json_path = os.path.join(run_path, "hyperparams.json")
        if not os.path.isfile(json_path):
            continue

        with open(json_path, "r") as f:
            hp = json.load(f)

        seed = hp["seed"]
        seed_folder = f"seed_{seed}"

        # Find weight file
        weight_file = next(
            (os.path.join(run_path, f)
            for f in os.listdir(run_path)
            if f.endswith(".pt") or f.endswith(".pth")),
            None
        )
        if weight_file is None:
            continue

        out_dir = os.path.join(output_root, seed_folder, constraint)
        os.makedirs(out_dir, exist_ok=True)

        ext = os.path.splitext(weight_file)[1]
        out_name = f"{constraint}_{seed}{ext}"
        out_path = os.path.join(out_dir, out_name)
        shutil.copy2(weight_file, out_path)

        print(f"Copied: seed_{seed} | {constraint} | {out_name}")

In [ ]:
# STEP 2: MOVING ORGANIZED LOGS TO FINAL MODEL WEIGHTS BY PROTOCOL
root = "model_weights/training_logs"
out_root = "model_weights"

mapping = {
    "development/datasets/non_isothermal/binary/combined_binary.csv"            : "protocol_I",
}

for seed_folder in os.listdir(root):
    seed_path = os.path.join(root, seed_folder)
    if not os.path.isdir(seed_path):
        continue
    seed_num = seed_folder.split("_")[1]
    for constraint in ["hard_constraint", "soft_constraint", "none_constraint"]:
        constraint_path = os.path.join(seed_path, constraint)
        if not os.path.isdir(constraint_path):
            continue
        for f in os.listdir(constraint_path):
            if not (f.endswith(".pt") or f.endswith(".pth")):
                continue
            src = os.path.join(constraint_path, f)
            # You'll need to determine protocol some other way since there's no hyperparams.json
            # For now, defaulting to protocol_I — adjust as needed
            protocol = "protocol_I"
            out_dir = os.path.join(out_root, protocol, constraint)
            os.makedirs(out_dir, exist_ok=True)
            ext = os.path.splitext(f)[1]
            out_name = f"{constraint}_{seed_num}{ext}"
            out_path = os.path.join(out_dir, out_name)
            shutil.copy2(src, out_path)
            print(f"Extracted: {protocol:<12} | {constraint:<15} | seed {seed_num:>4}")